# ComeHear ASR v2: Fine-tuning Whisper untuk Anamnesis Poli Anak (Indonesia + Jawa + istilah medis)

Lanjutan `train-java-indo-voice.ipynb` (v1: whisper-small, WER 34,55% tanpa normalisasi). Perubahan v2:

| Masalah v1 | Perbaikan v2 |
|---|---|
| WER dihitung hanya `lower()`; tanda baca dan angka dihitung salah | Normalisasi teks yang sama dengan backend (`app/scribe/textnorm.py`) |
| Tidak ada pembanding | WER **baseline** (sebelum fine-tune) vs **sesudah**, per domain |
| Tidak ada data percakapan medis | Opsional: kalimat klinis sintetis (TTS) + rekaman role-play tim |
| whisper-small | Pilihan model: `whisper-large-v3-turbo` (default) / `whisper-medium` / `whisper-small` |
| LoRA hanya q,v | LoRA q,k,v,out + SpecAugment |
| Checkpoint terbaik dipilih dengan WER mentah | Dipilih dengan WER ternormalisasi |

Target proposal: WER ≤ 15–20%. Jalankan di Kaggle (GPU T4/P100). Isi bagian **KONFIGURASI** dulu.

In [ ]:
# [1] Instalasi
!pip install -q --upgrade pip
!pip uninstall -y -q torchao
!pip install -q transformers peft datasets evaluate jiwer accelerate soundfile librosa ctranslate2 faster-whisper

In [ ]:
# [2] KONFIGURASI
QUICK_TRIAL = True            # True: ~200 sampel, 60 langkah untuk cek alur. False: pelatihan penuh.
MODEL_ID = "openai/whisper-large-v3-turbo"   # alternatif: "openai/whisper-medium", "openai/whisper-small"
N_INDONESIAN = 4000           # sampel Common Voice ID (penuh)
USE_SYNTHETIC_MEDICAL = True  # kalimat klinis sintetis lewat TTS (lihat sel 5)
N_SYNTHETIC = 1200
TEAM_RECORDINGS_DIR = "/kaggle/input/comehear-roleplay"  # opsional: folder berisi metadata.csv (file,sentence)
LEARNING_RATE = 1e-4
MAX_STEPS = 60 if QUICK_TRIAL else 2000
EVAL_STEPS = 30 if QUICK_TRIAL else 250
BATCH = 4 if "large" in MODEL_ID else 8
GRAD_ACCUM = 4 if "large" in MODEL_ID else 2
SEED = 42

In [ ]:
# [3] Import + normalisasi teks (salinan app/scribe/textnorm.py agar WER sama dengan backend)
import os, re, gc, glob, json, random, shutil, sys, unicodedata, inspect, warnings
from dataclasses import dataclass
import numpy as np, pandas as pd, torch, librosa, soundfile as sf
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import (WhisperProcessor, WhisperForConditionalGeneration,
                          Seq2SeqTrainingArguments, Seq2SeqTrainer)
from peft import LoraConfig, get_peft_model, PeftModel
warnings.filterwarnings("ignore")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")

_PUNCT = re.compile(r"[^\w\s]"); _SPACE = re.compile(r"\s+")
_ANGKA = {"nol":0,"satu":1,"dua":2,"tiga":3,"empat":4,"lima":5,"enam":6,"tujuh":7,"delapan":8,"sembilan":9,
          "sepuluh":10,"sebelas":11,"seratus":100}
_SINGKATAN = {"dok":"dokter","bu":"ibu","pak":"bapak","nggak":"tidak","gak":"tidak","ga":"tidak",
              "enggak":"tidak","udah":"sudah","aja":"saja","kalo":"kalau"}

def _words_to_digits(words):
    out, i = [], 0
    while i < len(words):
        w = words[i]
        if w in _ANGKA:
            n = _ANGKA[w]
            if i + 1 < len(words) and words[i+1] == "puluh":
                n, i = n * 10, i + 1
                if i + 1 < len(words) and words[i+1] in _ANGKA and _ANGKA[words[i+1]] < 10:
                    n, i = n + _ANGKA[words[i+1]], i + 1
            elif i + 1 < len(words) and words[i+1] == "belas":
                n, i = n + 10, i + 1
            out.append(str(n))
        else:
            out.append(w)
        i += 1
    return out

def normalize(t):
    t = unicodedata.normalize("NFKC", t).lower()
    t = re.sub(r"(\d),(\d)", r"\1 koma \2", t).replace("-", " ")
    words = _SPACE.sub(" ", _PUNCT.sub(" ", t)).strip().split()
    return " ".join(_words_to_digits([_SINGKATAN.get(w, w) for w in words]))

assert normalize("Suhu 38,6 derajat, Dok!") == "suhu 38 koma 6 derajat dokter"

In [ ]:
# [4] Data: Common Voice ID + Jawa-Lumajang + (opsional) rekaman role-play tim
KAGGLE = "/kaggle/input"
def valid(p, min_bytes=2048):
    try: return os.path.exists(p) and os.path.getsize(p) > min_bytes
    except OSError: return False

frames = []
cv = glob.glob(f"{KAGGLE}/**/train.tsv", recursive=True) or glob.glob(f"{KAGGLE}/**/validated.tsv", recursive=True)
if cv:
    root = os.path.dirname(cv[0])
    df = pd.read_csv(cv[0], sep="\t", low_memory=False)
    df["audio_path"] = df["path"].map(lambda x: os.path.join(root, "clips", str(x)))
    df = df[df["sentence"].notna() & df["audio_path"].map(valid)]
    df = df.sample(n=min(100 if QUICK_TRIAL else N_INDONESIAN, len(df)), random_state=SEED)
    frames.append(pd.DataFrame({"audio_path": df["audio_path"], "sentence": df["sentence"].str.strip(), "domain": "indonesia"}))

jw = glob.glob(f"{KAGGLE}/**/Mapping.tsv", recursive=True)
if jw:
    root = os.path.dirname(jw[0])
    df = pd.read_csv(jw[0], sep="\t")
    df["audio_path"] = df.apply(lambda r: os.path.join(root, str(r["speaker_number"]), str(r["audio_filename"])), axis=1)
    df = df[df["post-edited transcription"].notna() & df["audio_path"].map(lambda p: valid(p, 1024))]
    if QUICK_TRIAL: df = df.sample(n=min(50, len(df)), random_state=SEED)
    frames.append(pd.DataFrame({"audio_path": df["audio_path"], "sentence": df["post-edited transcription"].str.strip(), "domain": "jawa"}))

meta = os.path.join(TEAM_RECORDINGS_DIR, "metadata.csv")
if os.path.exists(meta):
    df = pd.read_csv(meta)
    df["audio_path"] = df["file"].map(lambda f: os.path.join(TEAM_RECORDINGS_DIR, f))
    df = df[df["audio_path"].map(valid)]
    frames.append(pd.DataFrame({"audio_path": df["audio_path"], "sentence": df["sentence"].str.strip(), "domain": "roleplay"}))
    print("Rekaman role-play tim:", len(df))

### Data klinis sintetis (opsional)
Common Voice dan dialek Lumajang tidak berisi percakapan dokter. Sel berikut membuat kalimat anamnesis poli anak
dari templat (istilah klinis, angka suhu/berat, dosis, campuran Jawa) lalu menyuarakannya dengan
`facebook/mms-tts-ind` plus derau ringan dan variasi kecepatan. Porsinya sengaja kecil (±20%) supaya model
belajar **kosakata** tanpa terlalu menyesuaikan diri ke suara TTS.

Catatan lisensi: MMS-TTS berlisensi CC-BY-NC 4.0 (non-komersial). Aman untuk prototipe lomba; untuk produk
SaaS ganti dengan rekaman role-play tim (`TEAM_RECORDINGS_DIR`) atau TTS berlisensi komersial.

In [ ]:
# [5] Kalimat klinis sintetis + TTS
KELUHAN = ["demam", "panas", "batuk berdahak", "batuk kering", "pilek", "mencret", "diare", "muntah", "mual",
           "sesak napas", "nyeri perut", "sakit kepala", "ruam merah", "biduran", "gatal-gatal", "kejang",
           "mimisan", "nafsu makan turun", "rewel", "lemas", "sumer", "ngelu", "batuk grok-grok"]
DURASI = ["sejak kemarin", "sejak tadi pagi", "sudah dua hari", "sudah tiga hari", "sudah seminggu",
          "ket wingi", "ket isuk", "naik turun sejak lusa"]
OBAT = ["parasetamol sirup", "ibuprofen sirup", "amoksisilin", "oralit", "zink", "salbutamol", "cetirizine",
        "tempra", "sanmol", "proris"]
ANGKA = ["tiga puluh delapan koma lima", "tiga puluh sembilan", "tiga puluh tujuh koma delapan",
         "empat puluh", "tiga puluh delapan koma sembilan"]
POLA = [
    "Anak saya {k} {d}, Dok.", "Ibu, adiknya {k} {d}?", "Sudah dikasih {o} di rumah, Dok.",
    "Suhunya tadi {a} derajat.", "Ada alergi obat atau makanan, Bu?", "Kalau makan udang dia biduran, Dok.",
    "Selain {k}, ada {k2} juga?", "Saya resepkan {o} lima mililiter tiap empat sampai enam jam kalau demam.",
    "Berat badannya {b} kilo.", "Minumnya berkurang, buang air kecil masih lancar.",
    "Pernah dirawat karena {k} sebelumnya?", "Awakke {k} {d}, Dok, terus ngelu.",
]
def synth_sentence():
    k, k2 = random.sample(KELUHAN, 2)
    b = random.choice(["sembilan", "dua belas", "lima belas", "delapan belas", "dua puluh dua"])
    return random.choice(POLA).format(k=k, k2=k2, d=random.choice(DURASI), o=random.choice(OBAT), a=random.choice(ANGKA), b=b)

if USE_SYNTHETIC_MEDICAL:
    from transformers import VitsModel, AutoTokenizer
    tts_tok = AutoTokenizer.from_pretrained("facebook/mms-tts-ind")
    tts = VitsModel.from_pretrained("facebook/mms-tts-ind").to(device).eval()
    out_dir = "/kaggle/working/synthetic"; os.makedirs(out_dir, exist_ok=True)
    rows = []
    for n in range(40 if QUICK_TRIAL else N_SYNTHETIC):
        text = synth_sentence()
        with torch.no_grad():
            wav = tts(**tts_tok(text, return_tensors="pt").to(device)).waveform[0].cpu().numpy()
        sr = tts.config.sampling_rate
        wav = librosa.effects.time_stretch(wav, rate=random.uniform(0.85, 1.15))
        wav = wav + np.random.normal(0, random.uniform(0.001, 0.01), wav.shape)  # derau ruang periksa
        path = f"{out_dir}/syn_{n:05d}.wav"
        sf.write(path, wav.astype(np.float32), sr)
        rows.append({"audio_path": path, "sentence": text, "domain": "medis_sintetis"})
    frames.append(pd.DataFrame(rows))
    del tts; gc.collect(); torch.cuda.empty_cache()

data = pd.concat(frames, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
print(data["domain"].value_counts())

In [ ]:
# [6] Split per domain: train / val (pilih checkpoint) / test (laporan akhir, tidak pernah dilihat saat latih)
train_df, rest = train_test_split(data, test_size=0.2, random_state=SEED, stratify=data["domain"])
val_df, test_df = train_test_split(rest, test_size=0.5, random_state=SEED, stratify=rest["domain"])
print(len(train_df), len(val_df), len(test_df))

processor = WhisperProcessor.from_pretrained(MODEL_ID, language="indonesian", task="transcribe")

def prepare(batch):
    try:
        audio, _ = librosa.load(batch["audio_path"], sr=16000)
    except Exception:
        audio = np.zeros(16000, dtype=np.float32)
    batch["input_features"] = processor.feature_extractor(audio, sampling_rate=16000).input_features[0]
    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids[:440]
    return batch

ds = DatasetDict({
    "train": Dataset.from_pandas(train_df[["audio_path", "sentence"]].reset_index(drop=True)),
    "val": Dataset.from_pandas(val_df[["audio_path", "sentence"]].reset_index(drop=True)),
}).map(prepare, remove_columns=["audio_path", "sentence"], writer_batch_size=100)
gc.collect()

@dataclass
class Collator:
    processor: object
    def __call__(self, features):
        batch = self.processor.feature_extractor.pad([{"input_features": f["input_features"]} for f in features], return_tensors="pt")
        labels = self.processor.tokenizer.pad([{"input_ids": f["labels"]} for f in features], return_tensors="pt")
        ids = labels["input_ids"].masked_fill(labels.attention_mask.ne(1), -100)
        if (ids[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            ids = ids[:, 1:]
        batch["labels"] = ids
        return batch

In [ ]:
# [7] Fungsi evaluasi WER/CER ternormalisasi per domain (dipakai untuk baseline dan hasil akhir)
import jiwer

def transcribe_df(model, df, batch=8):
    model.eval(); preds = []
    for i in range(0, len(df), batch):
        rows = df.iloc[i:i + batch]
        audios = [librosa.load(p, sr=16000)[0] for p in rows["audio_path"]]
        feats = processor.feature_extractor(audios, sampling_rate=16000, return_tensors="pt").input_features
        with torch.no_grad(), torch.autocast(device, dtype=torch.float16, enabled=device == "cuda"):
            ids = model.generate(input_features=feats.to(device, dtype=model.dtype), language="indonesian",
                                 task="transcribe", max_new_tokens=220, num_beams=1)
        preds += processor.batch_decode(ids, skip_special_tokens=True)
    return preds

def report(model, df, title):
    preds = transcribe_df(model, df)
    out = df.assign(pred=preds)
    rows = []
    for dom, g in list(out.groupby("domain")) + [("SEMUA", out)]:
        refs = [normalize(s) for s in g["sentence"]]; hyps = [normalize(p) for p in g["pred"]]
        keep = [i for i, r in enumerate(refs) if r]
        rows.append({"domain": dom, "n": len(keep),
                     "WER%": round(100 * jiwer.wer([refs[i] for i in keep], [hyps[i] for i in keep]), 2),
                     "CER%": round(100 * jiwer.cer([refs[i] for i in keep], [hyps[i] for i in keep]), 2)})
    res = pd.DataFrame(rows); print(f"\n== {title} =="); print(res.to_string(index=False))
    return res, out

base = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float16 if device == "cuda" else torch.float32).to(device)
baseline, _ = report(base, test_df if not QUICK_TRIAL else test_df.head(40), f"BASELINE {MODEL_ID} (tanpa fine-tune)")
del base; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# [8] LoRA + pelatihan (checkpoint terbaik dipilih dengan WER ternormalisasi)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model.config.apply_spec_augment = True
model.config.mask_time_prob = 0.05
model.config.use_cache = False
model.generation_config.language = "indonesian"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.forced_decoder_ids = None
model.config.suppress_tokens = None
model.config.begin_suppress_tokens = None
try:
    import peft.import_utils; peft.import_utils.is_torchao_available = lambda: False
except Exception:
    pass
model = get_peft_model(model, LoraConfig(r=32, lora_alpha=64, lora_dropout=0.05, bias="none",
                                         target_modules=["q_proj", "k_proj", "v_proj", "out_proj"]))
model.print_trainable_parameters()

def compute_metrics(pred):
    ids, labels = pred.predictions, pred.label_ids
    labels[labels == -100] = processor.tokenizer.pad_token_id
    p = [normalize(x) for x in processor.tokenizer.batch_decode(ids, skip_special_tokens=True)]
    r = [normalize(x) for x in processor.tokenizer.batch_decode(labels, skip_special_tokens=True)]
    keep = [i for i, x in enumerate(r) if x]
    return {"wer": 100 * jiwer.wer([r[i] for i in keep], [p[i] for i in keep]),
            "cer": 100 * jiwer.cer([r[i] for i in keep], [p[i] for i in keep])}

args = Seq2SeqTrainingArguments(
    output_dir="./asr-v2", per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE, warmup_steps=max(10, MAX_STEPS // 10), max_steps=MAX_STEPS, max_grad_norm=1.0,
    gradient_checkpointing=True, fp16=device == "cuda", eval_strategy="steps", eval_steps=EVAL_STEPS,
    save_steps=EVAL_STEPS, logging_steps=max(10, EVAL_STEPS // 5), per_device_eval_batch_size=BATCH,
    predict_with_generate=True, generation_max_length=225, report_to=[], load_best_model_at_end=True,
    metric_for_best_model="wer", greater_is_better=False, save_total_limit=2, remove_unused_columns=False,
    label_names=["labels"], seed=SEED)
kw = dict(args=args, model=model, train_dataset=ds["train"], eval_dataset=ds["val"],
          data_collator=Collator(processor), compute_metrics=compute_metrics)
params = inspect.signature(Seq2SeqTrainer.__init__).parameters
kw["processing_class" if "processing_class" in params else "tokenizer"] = processor.feature_extractor
trainer = Seq2SeqTrainer(**kw)
trainer.train()
model.save_pretrained("./asr-v2-lora"); processor.save_pretrained("./asr-v2-lora")

In [ ]:
# [9] Gabung LoRA, evaluasi di TEST set, bandingkan dengan baseline
base = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
merged = PeftModel.from_pretrained(base, "./asr-v2-lora").merge_and_unload()
merged.generation_config.forced_decoder_ids = None
merged.save_pretrained("./asr-v2-merged"); processor.save_pretrained("./asr-v2-merged")
merged = merged.to(device).half() if device == "cuda" else merged.to(device)
final, preds = report(merged, test_df if not QUICK_TRIAL else test_df.head(40), "SESUDAH FINE-TUNE (test set)")
cmp = baseline.merge(final, on=["domain", "n"], suffixes=(" baseline", " v2"))
print("\n== PERBANDINGAN =="); print(cmp.to_string(index=False))
cmp.to_csv("wer_report.csv", index=False)
preds.to_csv("test_predictions.csv", index=False)   # bahan analisis kesalahan istilah medis

In [ ]:
# [10] Ekspor CTranslate2 untuk backend (ASR_MODEL=<folder> di .env), dikompres jadi zip
import ctranslate2.converters.transformers as ct2
CT2 = "./comehear-asr-v2-ct2"
old = sys.argv
try:
    sys.argv = ["ct2", "--model", "./asr-v2-merged", "--output_dir", CT2, "--quantization", "float16", "--force",
                "--copy_files", "tokenizer.json", "preprocessor_config.json"]
    ct2.main()
finally:
    sys.argv = old
shutil.make_archive("comehear-asr-v2-ct2", "zip", CT2)
print("Unduh comehear-asr-v2-ct2.zip dari tab Output, ekstrak ke comehear/models/, lalu set ASR_MODEL di .env")

In [ ]:
# [11] Uji cepat faster-whisper (sama persis dengan backend: CPU int8 bila tanpa GPU)
from faster_whisper import WhisperModel
fw = WhisperModel(CT2, device=device, compute_type="float16" if device == "cuda" else "int8")
for _, r in test_df.head(5).iterrows():
    segs, _ = fw.transcribe(r["audio_path"], language="id", beam_size=5, condition_on_previous_text=False)
    hyp = " ".join(s.text for s in segs).strip()
    print(f"[{r['domain']}] REF: {r['sentence']}\n           HYP: {hyp}\n")